In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW  # берём из PyTorch
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd

/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# --- Параметры ---
MODEL_NAME = "roberta-base"
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Данные ---
# X = список текстов
# y = numpy array shape (num_samples, num_labels) с 0/1
# label_cols = список названий колонок
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels.values if isinstance(labels, pd.DataFrame) else labels
        self.tokenizer = tokenizer
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        tokens = self.tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in tokens.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

# --- Модель ---
class RobertaMultiLabel(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_labels)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:,0,:]  # CLS токен
        dropped = self.dropout(pooled)
        return torch.sigmoid(self.classifier(dropped))



In [3]:
df=pd.read_csv("new_ds.csv")

In [4]:
df.head(5)

,text,ASSORTMENT,PROMOTIONS,DELIVERY,PRICE,PRODUCTS_QUALITY,SUPPORT,CATALOG_NAVIGATION,PAYMENT
0,"Маленький выбор товаров, хотелось бы ассортиме...",1,0,0,0,0,0,0,0
1,Быстро,0,0,1,0,0,0,0,0
2,Доставка постоянно задерживается,0,0,1,0,0,0,0,0
3,Наценка и ассортимент расстраивают,1,0,0,1,0,0,0,0
4,Можно немного скинуть минимальную сумму заказа...,0,0,1,1,0,0,0,1


In [5]:
df[df.columns[1:]]=df[df.columns[1:]].astype("int")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2282 entries, 0 to 2281
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   text                2282 non-null   str  
 1   ASSORTMENT          2282 non-null   int64
 2   PROMOTIONS          2282 non-null   int64
 3   DELIVERY            2282 non-null   int64
 4   PRICE               2282 non-null   int64
 5   PRODUCTS_QUALITY    2282 non-null   int64
 6   SUPPORT             2282 non-null   int64
 7   CATALOG_NAVIGATION  2282 non-null   int64
 8   PAYMENT             2282 non-null   int64
dtypes: int64(8), str(1)
memory usage: 502.5 KB


In [6]:
X=df['text']
y=df.drop(columns=["text"])

In [7]:
# --- Подготовка ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
dataset = TextDataset(X, y, tokenizer)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = RobertaMultiLabel(MODEL_NAME, y.shape[1]).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=1e-5)
criterion = nn.BCELoss()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 15540.30it/s]
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
for epoch in range(EPOCHS):
    model.train()
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} loss: {loss.item():.4f}")

# --- Предсказание ---
model.eval()
preds = []
with torch.no_grad():
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        outputs = model(input_ids, attention_mask)
        preds.append(outputs.cpu().numpy())
preds = np.vstack(preds)


Epoch 1 loss: 0.3212
Epoch 2 loss: 0.1886
Epoch 3 loss: 0.4271
Epoch 4 loss: 0.3840
Epoch 5 loss: 0.2015
Epoch 6 loss: 0.1709
Epoch 7 loss: 0.1597
Epoch 8 loss: 0.1419
Epoch 9 loss: 0.1808
Epoch 10 loss: 0.1116
Epoch 11 loss: 0.2382
Epoch 12 loss: 0.1970
Epoch 13 loss: 0.2270
Epoch 14 loss: 0.0748
Epoch 15 loss: 0.0792
Epoch 16 loss: 0.0815
Epoch 17 loss: 0.1331
Epoch 18 loss: 0.1306
Epoch 19 loss: 0.1781
Epoch 20 loss: 0.0641
Epoch 21 loss: 0.1219
Epoch 22 loss: 0.1300
Epoch 23 loss: 0.1470
Epoch 24 loss: 0.1374
Epoch 25 loss: 0.1072
Epoch 26 loss: 0.0760
Epoch 27 loss: 0.0718
Epoch 28 loss: 0.0458
Epoch 29 loss: 0.1130
Epoch 30 loss: 0.1281
Epoch 31 loss: 0.0335
Epoch 32 loss: 0.0345
Epoch 33 loss: 0.1034
Epoch 34 loss: 0.0576
Epoch 35 loss: 0.0717
Epoch 36 loss: 0.0253
Epoch 37 loss: 0.0387
Epoch 38 loss: 0.0327
Epoch 39 loss: 0.0571
Epoch 40 loss: 0.0260
Epoch 41 loss: 0.0430
Epoch 42 loss: 0.0153
Epoch 43 loss: 0.0286
Epoch 44 loss: 0.0183
Epoch 45 loss: 0.0160
Epoch 46 loss: 0.02

In [9]:
preds

array([[1.17364398e-03, 3.47989990e-04, 9.99438941e-01, ...,
        5.77112602e-04, 4.75851150e-04, 1.06791995e-04],
       [3.87285982e-04, 2.64377450e-04, 9.67144500e-04, ...,
        5.30983321e-04, 6.83244143e-04, 1.86991223e-04],
       [3.68080306e-04, 2.24789648e-04, 9.99277651e-01, ...,
        3.50235525e-04, 2.34340841e-04, 1.93605985e-04],
       ...,
       [4.35206137e-04, 2.51610094e-04, 9.99533772e-01, ...,
        3.70684545e-04, 1.65872305e-04, 2.45534466e-04],
       [9.98566329e-01, 9.99060929e-01, 4.49678907e-03, ...,
        9.71273612e-03, 6.51893159e-03, 9.68383439e-03],
       [4.19293385e-04, 4.48307052e-04, 2.61505513e-04, ...,
        2.60006404e-04, 5.01444738e-04, 3.65287968e-04]],
      shape=(2282, 8), dtype=float32)

In [10]:
# --- Применяем разные пороги для каждого класса ---
thresholds = [0.10, 0.001, 0.8, 0.36, 0.4, 0.1, 0.001, 0.001]

preds_binary = np.zeros_like(preds, dtype=int)
for i, t in enumerate(thresholds):
    preds_binary[:, i] = (preds[:, i] > t).astype(int)

In [11]:
preds_binary

array([[0, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 1, ..., 0, 0, 0],
       [1, 1, 0, ..., 0, 1, 1],
       [0, 0, 0, ..., 0, 0, 0]], shape=(2282, 8))

In [13]:
from sklearn.metrics import classification_report


print(classification_report(y,preds_binary))

              precision    recall  f1-score   support

           0       0.12      0.12      0.12       257
           1       0.04      0.37      0.07        95
           2       0.54      0.54      0.54      1234
           3       0.19      0.19      0.19       401
           4       0.16      0.16      0.16       433
           5       0.11      0.11      0.11       263
           6       0.06      0.41      0.11       137
           7       0.03      0.66      0.05        38

   micro avg       0.18      0.35      0.24      2858
   macro avg       0.16      0.32      0.17      2858
weighted avg       0.31      0.35      0.31      2858
 samples avg       0.23      0.30      0.23      2858



/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, m